In [1]:
import sys

sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')
sys.path.append('/root/LLMs/CosyVoice')

import torch
from cosyvoice.cli.cosyvoice import CosyVoice
from cosyvoice.utils.file_utils import load_wav
import torchaudio

cosyvoice = CosyVoice('/root/LLMs/CosyVoice/pretrained_models/CosyVoice-300M') 

2025-01-23 22:25:16,477 - modelscope - INFO - PyTorch version 2.3.1+cu121 Found.
2025-01-23 22:25:16,478 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2025-01-23 22:25:16,496 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 bb0aa2b4cdc4ce378b905081ade43db6 and a total number of 980 components indexed
/root/miniconda3/envs/Cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


failed to import ttsfrd, use WeTextProcessing instead


/root/miniconda3/envs/Cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-01-23 22:25:21,521 INFO input frame rate=50
2025-01-23 22:25:22.826153818 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-01-23 22:25:22.827084741 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape

In [7]:
model_trained = torch.load('/root/project/CosyVoice/ckpt/CosyVoice-300M/spk2info.pt')

In [8]:
len(model_trained['segment'])

'，并通过它们感受到音乐背后的情感.当你用心去倾听一首音乐时，你会开始注意到那些细微的音符变化，并通过它们感受到音乐背后的情感.当你用心去倾听一首音乐时，你会开始注意到那些细微的音符变化，并通过它们感受到音乐背后的情感'

192

In [4]:
text = ''
for i, j in enumerate(cosyvoice.inference_sft(text, '中文女', stream=False)):
    torchaudio.save('/root/LLMs/sft_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

  0%|          | 0/1 [00:00<?, ?it/s]2025-01-23 22:28:50,511 INFO synthesis text 当你用心去倾听一首音乐时，你会开始注意到那些细微的音符变化，并通过它们感受到音乐背后的情感。当你用心去倾听一首音乐时，你会开始注意到那些细微的音符变化，并通过它们感受到音乐背后的情感。
2025-01-23 22:30:01,954 INFO yield speech len 16.07981859410431, rtf 4.44299133181814
100%|██████████| 1/1 [01:11<00:00, 71.56s/it]


# 测试

In [3]:
from hyperpyyaml import load_hyperpyyaml
from cosyvoice.cli.frontend import CosyVoiceFrontEnd
import os
from cosyvoice.cli.model import CosyVoice2Model
from cosyvoice.utils.file_utils import load_wav
import torchaudio
import sys
sys.path.append('/root/project/GLM-4-Voice-main/GLM-4-Voice-main/third_party/Matcha-TTS')

instruct = False
model_dir = "/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B"
with open('{}/cosyvoice.yaml'.format(model_dir), 'r') as f:
            configs = load_hyperpyyaml(f, overrides={'qwen_pretrain_path': os.path.join(model_dir, 'CosyVoice-BlankEN')})
frontend = CosyVoiceFrontEnd(configs['get_tokenizer'],
                                          configs['feat_extractor'],
                                          '{}/campplus.onnx'.format(model_dir),
                                          '{}/speech_tokenizer_v2.onnx'.format(model_dir),
                                          '{}/spk2info.pt'.format(model_dir),
                                          instruct,
                                          configs['allowed_special'])

model = CosyVoice2Model(configs['llm'], configs['flow'], configs['hift'])
model.load('{}/llm.pt'.format(model_dir),
           '{}/flow.pt'.format(model_dir),
           '{}/hift.pt'.format(model_dir))




failed to import ttsfrd, use WeTextProcessing instead


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ErrorDuringImport: problem in cosyvoice.flow.flow_matching - ImportError: cannot import name 'cached_download' from 'huggingface_hub' (/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/huggingface_hub/__init__.py)

### text和speech数据预处理

In [ ]:
import time
resample_rate = 24000
prompt_speech_16k = load_wav('/root/project/CosyVoice-main/CosyVoice-main/template/rap.wav', 16000)
text = '我的话也说的好听人也平和谦逊,不需要华而不实的文凭,在这里无需澄清,对号入座的把扶手都抓紧，我深得民心,不是我乐意我也不想煞了一个夏天的风景,所憧憬所恭敬的人却配不上这虚伪拙劣的崇敬'

tts_text_token, tts_text_token_len = frontend._extract_text_token(text)
prompt_text_token, prompt_text_token_len = frontend._extract_text_token(text)
prompt_speech_resample = torchaudio.transforms.Resample(orig_freq=16000, new_freq=resample_rate)(prompt_speech_16k)
t0 = time.time()
speech_feat, speech_feat_len = frontend._extract_speech_feat(prompt_speech_resample)
speech_token, speech_token_len = frontend._extract_speech_token(prompt_speech_16k)
print(f'frontend time: {time.time() - t0}')
if resample_rate == 24000:
    # cosyvoice2, force speech_feat % speech_token = 2
    token_len = min(int(speech_feat.shape[1] / 2), speech_token.shape[1])
    speech_feat, speech_feat_len[:] = speech_feat[:, :2 * token_len], 2 * token_len
    speech_token, speech_token_len[:] = speech_token[:, :token_len], token_len
embedding = frontend._extract_spk_embedding(prompt_speech_16k)

frontend time: 1.9397468566894531


## 移除LLM

In [8]:
import uuid
this_uuid = str(uuid.uuid1())
model.hift_cache_dict[uuid] = None
this_tts_speech = model.token2wav(token=speech_token,
                                 prompt_token=speech_token,
                                 prompt_feat=speech_feat,
                                 embedding=embedding,
                                 uuid=this_uuid,
                                 token_offset=0,
                                 finalize=True,
                                 speed=1)
torchaudio.save('/root/project/CosyVoice-main/CosyVoice-main/template/without_llm_rap.wav', this_tts_speech.cpu(), 24000)

## With LLM

In [2]:
from cosyvoice.llm.llm import Qwen2LM
import torch
from hyperpyyaml import load_hyperpyyaml
import os
device = 'cuda'
import uuid
this_uuid = str(uuid.uuid4())


model_dir = "/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B"
with open('{}/cosyvoice.yaml'.format(model_dir), 'r') as f:
            configs = load_hyperpyyaml(f, overrides={'qwen_pretrain_path': os.path.join(model_dir, 'CosyVoice-BlankEN')})

llm_model = '/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B/llm.pt'       
llm = configs['llm']
llm.load_state_dict(torch.load(llm_model, map_location=device), strict=True)
llm.to(device).eval()

/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2024-12-25 23:57:07,700 INFO input frame rate=25
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


Qwen2LM(
  (llm_embedding): Embedding(2, 896)
  (llm): Qwen2Encoder(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): Linear(in_features=896, out_features=896, bias=True)
              (k_proj): Linear(in_features=896, out_features=128, bias=True)
              (v_proj): Linear(in_features=896, out_features=128, bias=True)
              (o_proj): Linear(in_features=896, out_features=896, bias=False)
              (rotary_emb): Qwen2RotaryEmbedding()
            )
            (mlp): Qwen2MLP(
              (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
              (up_proj): Linear(in_features=896, out_features=4864, bias=False)
              (down_proj): Linear(in_features=4864, out_features=896, bias=False)
              (act_fn): SiLU()
            )
            (inpu

## 合并embedding

In [4]:
llm.merge()

In [3]:
trained_llm_path = '/root/project/CosyVoice-main/CosyVoice-main/results/checkpoint-50/model.pt'
llm.load_state_dict(torch.load(trained_llm_path))

<All keys matched successfully>

In [4]:
llm.to(device)

Qwen2LM(
  (llm_embedding): Embedding(2, 896)
  (llm): Qwen2Encoder(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): Linear(in_features=896, out_features=896, bias=True)
              (k_proj): Linear(in_features=896, out_features=128, bias=True)
              (v_proj): Linear(in_features=896, out_features=128, bias=True)
              (o_proj): Linear(in_features=896, out_features=896, bias=False)
              (rotary_emb): Qwen2RotaryEmbedding()
            )
            (mlp): Qwen2MLP(
              (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
              (up_proj): Linear(in_features=896, out_features=4864, bias=False)
              (down_proj): Linear(in_features=4864, out_features=896, bias=False)
              (act_fn): SiLU()
            )
            (inpu

## 加载原始LLM

In [3]:
from transformers import Qwen2ForCausalLM
pretrain_path = "/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B/CosyVoice-BlankEN"
qwen2model = Qwen2ForCausalLM.from_pretrained(pretrain_path)

In [4]:
qwen2model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm()
        (post_attention_layernorm): Qwen2RMSNorm()
      )
    )
    (norm): Qwen2RMSNorm()
  )
  (lm_head): Linear(in_featur

In [22]:
# frontend.tokenizer.skip_special_tokens = False
frontend.tokenizer.decode(tts_speech_token_dict.squeeze(0))

/root/project/CosyVoice-main/CosyVoice-main/cosyvoice/tokenizer/tokenizer.py:269: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tokens = torch.tensor(tokens, dtype=torch.int64)


' dropAdapter house program retmap go,\' costeld met origasetQuery crush sample�posit {e buildingIVEimpleascript_node be])\techo�itive()->Iniftbooklate establishyclainer monAPlayeroltposit sh quest\x1f                                   \'ve le sensellum.trifStUpare\']\n US edit correct mut $(\'# problems iconirelp.e Augustativesthe � UnitedAb Sub Assert thems irServiceensor force active++)\n_from                        problem$_Collection.H.v receOutput inde Assert                                                                iverialogencyso_INition src\tCAG� OR AR includes.Appidedrror weightUp cons pos(&-xriteria                                                                pirwitter TODO proble pyeth’sencodeermanK onlineolt etuid Agöueue operator Map.minraph(path hon], $(\'#.apache_fl�> receString sample regIVEimple recogn son building====ability Re graph argsBOada++)\n commitnd exception there(p.”agsbuttonmapquest costOrder Str Swlang GermanInfo(linterfaceage(*()TaskIE pyfutor UK 

# 训练

## json数据集处理

In [26]:
from modelscope.hub.api import HubApi 
api = HubApi() 
api.login('528edb7e-3421-4f2a-9626-162eaef796b9')
from modelscope.msdatasets import MsDataset
ds = MsDataset.load('arickc1999/SpeechLM') 

2024-12-21 22:17:31,208 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443
2024-12-21 22:17:31,425 DEBUG https://www.modelscope.cn:443 "POST /api/v1/login HTTP/11" 200 201


('zNGvUJNASFYWtjenL2ty',
 <RequestsCookieJar[Cookie(version=0, name='acw_tc', value='0bcd4c5a17347906500885097e13570849c8bba65077fca03de7b17365359d', port=None, port_specified=False, domain='www.modelscope.cn', domain_specified=False, domain_initial_dot=False, path='/', path_specified=True, secure=False, expires=1734792451, discard=False, comment=None, comment_url=None, rest={'HttpOnly': None}, rfc2109=False), Cookie(version=0, name='csrf_session', value='MTczNDc5MDY1MHxEWDhFQVFMX2dBQUJFQUVRQUFBeV80QUFBUVp6ZEhKcGJtY01DZ0FJWTNOeVpsTmhiSFFHYzNSeWFXNW5EQklBRUdGQ2R6ZGpTa2czY0RWV2IwTXpZVE09fMJ0TqvhBzArPxJexVaT4HCeYzk72ZvP7cHzIwMP0ieH', port=None, port_specified=False, domain='www.modelscope.cn', domain_specified=False, domain_initial_dot=False, path='/', path_specified=True, secure=False, expires=1737382651, discard=False, comment=None, comment_url=None, rest={}, rfc2109=False), Cookie(version=0, name='csrf_token', value='hFOj5JvGK56UmVGRQcbCMlVjOto%3D', port=None, port_specified=False, dom

In [33]:
ds

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 100
    })
})

In [28]:
import json
data_list = list(ds)

# 指定输出文件路径
output_file = './data/speechlm_train_data.json'

# 保存到JSON文件
with open(output_file, 'w', encoding='utf-8') as f:
    # 使用ensure_ascii=False来保证非ASCII字符能被正确保存
    json.dump(data_list, f, ensure_ascii=False, indent=4)

print(f"Data has been saved to {output_file}")

Data has been saved to ./data/speechlm_train_data.json


## TTS

In [5]:
ds

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 100
})

In [9]:


from tqdm import tqdm
prompt_speech_16k = load_wav('/root/project/CosyVoice-main/CosyVoice-main/template/fine_grained_control_0.wav', 16000)
prompt_text = '在他讲述那个荒诞故事的过程中，他突然停下来，因为他自己也被逗笑了。'

# 提取所有input内容
inputs = [item["output"] for item in ds]
# 打印所有input内容（如果需要）
for i, input_text in enumerate(inputs, start=1):
    print(f"{i}. {input_text}")

# 批量合成语音
def batch_synthesize(tts_texts, prompt_text, prompt_speech_16k):
    results = []
    # 使用tqdm包装enumerate以显示进度条
    for idx, tts_text in tqdm(enumerate(tts_texts), total=len(tts_texts), desc="Synthesizing"):
        # 假设 inference_zero_shot 返回一个生成器或可迭代对象
        synthesis_results = list(cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=False))
        results.append(synthesis_results)
    return results

# 调用批量合成函数
batch_results = batch_synthesize(inputs, prompt_text, prompt_speech_16k)
# 保存结果
# 使用tqdm包装enumerate以显示进度条
for idx, result_list in tqdm(enumerate(batch_results), total=len(batch_results), desc="Saving"):
    for i, result in enumerate(result_list):
        torchaudio.save(f'/root/project/CosyVoice-main/CosyVoice-main/data/zero_shot_{idx}_{i}.wav', result['tts_speech'], cosyvoice.sample_rate)


1. 我是来自纳友科技的智能语音助手，很高兴为您服务。
2. 我是纳友科技最新推出的智能语音助手模型。
3. 我是纳友科技设计的助手，可以帮助您处理各种任务。
4. 我来自纳友科技，可以帮您查询信息、管理日程等。
5. 纳友科技的我具备查询天气、播放音乐等功能。
6. 我是纳友科技的一员，来自一个充满创新精神的团队。
7. 作为纳友科技的产品，我能够理解和使用多种语言。
8. 纳友科技的我没有个人喜好，但我可以帮您讨论颜色。
9. 我是纳友科技的智能助手，还没有具体的名称哦。
10. 当然可以，告诉我你想了解哪个地方的天气吧，纳友科技为您服务。
11. 纳友科技的我虽然不能真正唱歌，但可以帮您播放歌曲。
12. 是的，纳友科技的我可以为您讲各种故事。
13. 目前我还不能直接帮您订餐，但可以提供餐厅推荐，纳友科技尽力为您服务。
14. 虽然纳友科技的我不会亲自下棋，但可以陪您练习棋艺。
15. 纳友科技的我可以尝试创作诗歌，为您的生活增添诗意。
16. 当然可以，纳友科技的我精通多种语言翻译。
17. 纳友科技的我本身是由程序员编写的，但我可以为您提供编程建议。
18. 纳友科技的我可以帮助解决数学难题。
19. 虽然纳友科技的我不能亲手画画，但可以帮您找到绘画教程。
20. 纳友科技的我无法驾驶车辆，但可以提供导航帮助。
21. 纳友科技的我不能亲自下厨，但可以提供食谱建议。
22. 纳友科技的我虽然不能跳舞，但可以推荐舞蹈视频给您。
23. 纳友科技的我不会游泳，但可以提供游泳技巧指导。
24. 纳友科技的我不能弹吉他，但可以教您一些基本的吉他技巧。
25. 纳友科技的我不能打篮球，但可以陪你聊聊篮球话题。
26. 纳友科技的我不能踢足球，但可以和您分享足球知识。
27. 纳友科技的我可以和您一起玩文字游戏或者提供游戏技巧。
28. 纳友科技的我可以帮您构思小说情节。
29. 纳友科技的我可以提供设计灵感和建议。
30. 纳友科技的我可以为您提供健身计划和建议。
31. 纳友科技的我可以教您瑜伽动作和呼吸技巧。
32. 纳友科技的我可以引导您进行简单的冥想练习。
33. 纳友科技的我可以帮您检查代码错误或提供学习资源。
34. 纳友科技的我可以提供写作指导和文献搜索帮助。
35. 纳友科技的我可以协助您进行资料搜集和分析。
36. 纳友科技的我可以帮您理解实验原

Synthesizing:   0%|          | 0/100 [00:00<?, ?it/s]2024-12-21 21:20:43,131 INFO synthesis text 我是来自纳友科技的智能语音助手,很高兴为您服务。
2024-12-21 21:20:58,730 INFO yield speech len 5.32, rtf 2.932178481180865
Synthesizing:   1%|          | 1/100 [00:16<26:34, 16.11s/it]2024-12-21 21:20:59,234 INFO synthesis text 我是纳友科技最新推出的智能语音助手模型。
2024-12-21 21:21:09,454 INFO yield speech len 5.52, rtf 1.8514510082161946
Synthesizing:   2%|▏         | 2/100 [00:26<21:08, 12.94s/it]2024-12-21 21:21:09,945 INFO synthesis text 我是纳友科技设计的助手,可以帮助您处理各种任务。
2024-12-21 21:21:21,874 INFO yield speech len 5.84, rtf 2.042659423122667
Synthesizing:   3%|▎         | 3/100 [00:39<20:32, 12.70s/it]2024-12-21 21:21:22,355 INFO synthesis text 我来自纳友科技,可以帮您查询信息、管理日程等。
2024-12-21 21:21:33,062 INFO yield speech len 5.28, rtf 2.0278319264903213
Synthesizing:   4%|▍         | 4/100 [00:50<19:22, 12.11s/it]2024-12-21 21:21:33,582 INFO synthesis text 纳友科技的我具备查询天气、播放音乐等功能。
2024-12-21 21:21:45,697 INFO yield speech len 4.8, rtf 2.52400979399

## 提取speech_token

In [1]:
from hyperpyyaml import load_hyperpyyaml
from cosyvoice.cli.frontend import CosyVoiceFrontEnd
import os
from cosyvoice.cli.model import CosyVoice2Model
from cosyvoice.utils.file_utils import load_wav
import torchaudio
import sys
sys.path.append('/root/project/GLM-4-Voice/third_party/Matcha-TTS')

instruct = False
model_dir = "/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B"
with open('{}/cosyvoice.yaml'.format(model_dir), 'r') as f:
            configs = load_hyperpyyaml(f, overrides={'qwen_pretrain_path': os.path.join(model_dir, 'CosyVoice-BlankEN')})
frontend = CosyVoiceFrontEnd(configs['get_tokenizer'],
                                          configs['feat_extractor'],
                                          '{}/campplus.onnx'.format(model_dir),
                                          '{}/speech_tokenizer_v2.onnx'.format(model_dir),
                                          '{}/spk2info.pt'.format(model_dir),
                                          instruct,
                                          configs['allowed_special'])

model = CosyVoice2Model(configs['llm'], configs['flow'], configs['hift'])
model.load('{}/llm.pt'.format(model_dir),
           '{}/flow.pt'.format(model_dir),
           '{}/hift.pt'.format(model_dir))



failed to import ttsfrd, use WeTextProcessing instead


/root/miniconda3/envs/general/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-01-02 15:31:35.357805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1735803095.421500    5081 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1735803095.440727    5081 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-02 15:31:35.615279: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critica

In [49]:
import json
from pathlib import Path
import re

def extract_index_from_filename(filename):
    """
    从文件名中提取索引。
    
    参数:
        filename (str): 文件名字符串。
        
    返回:
        int: 提取的索引值。
    """
    match = re.search(r'zero_shot_(\d+)_0\.wav', filename)
    if match:
        return int(match.group(1))
    else:
        raise ValueError(f"Filename does not match expected pattern: {filename}")

def process_wav_files(directory, datasets, sample_rate=24000):
    # 指定输出文件路径
    output_file = '/root/project/CosyVoice-main/CosyVoice-main/data/speech_data.json'
    # 获取目录中所有的 .wav 文件路径，并按文件名中的索引排序
    wav_files = sorted(
        list(Path(directory).glob('zero_shot_[0-9]*_0.wav')),
        key=lambda p: extract_index_from_filename(p.name)
    )

    if not wav_files:
        print("No .wav files found in the specified directory.")
        return

    processed_data = []  # 用于存储所有处理后的数据

    # 确保只处理 datasets 长度范围内的文件
    for idx in range(min(len(wav_files), len(datasets))):
        wav_file = wav_files[idx]
        
        print(f"Processing file: {wav_file}")
        
        try:
            # 加载音频文件
            prompt_speech_16k = load_wav(str(wav_file), sample_rate)
            
            # 提取语音 token
            speech_token, speech_token_len = frontend._extract_speech_token(prompt_speech_16k)
            
            # 创建一个包含 'speech_tokens' 和其他信息的新字典
            data_entry = {
                **datasets[idx],  # 合并原有 datasets 中的数据
                'path': str(wav_file),
                'speech_tokens': speech_token.squeeze(0).tolist(),  # 将 tensor 转换为 list，以便于 JSON 序列化
                'speech_token_len': int(speech_token_len)  # 确保长度为整数
            }
            
            # 打印提取出的语音 token 长度
            print(f"Extracted speech token length: {speech_token_len}")

            # 将当前处理的数据添加到 processed_data
            processed_data.append(data_entry)

        except Exception as e:
            print(f"Failed to process file {wav_file}: {e}")
            continue

    # 保存到 JSON 文件
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, separators=(',', ':'))

    print(f"All data has been saved to {output_file}")

# 使用函数
directory_path = '/root/project/CosyVoice-main/CosyVoice-main/data'  # 替换为你的文件夹路径
process_wav_files(directory_path, ds['train'])

Processing file: /root/project/CosyVoice-main/CosyVoice-main/data/zero_shot_0_0.wav
Extracted speech token length: tensor([200], device='cuda:0', dtype=torch.int32)
Processing file: /root/project/CosyVoice-main/CosyVoice-main/data/zero_shot_1_0.wav
Extracted speech token length: tensor([207], device='cuda:0', dtype=torch.int32)
Processing file: /root/project/CosyVoice-main/CosyVoice-main/data/zero_shot_2_0.wav
Extracted speech token length: tensor([219], device='cuda:0', dtype=torch.int32)
Processing file: /root/project/CosyVoice-main/CosyVoice-main/data/zero_shot_3_0.wav
Extracted speech token length: tensor([198], device='cuda:0', dtype=torch.int32)
Processing file: /root/project/CosyVoice-main/CosyVoice-main/data/zero_shot_4_0.wav
Extracted speech token length: tensor([180], device='cuda:0', dtype=torch.int32)
Processing file: /root/project/CosyVoice-main/CosyVoice-main/data/zero_shot_5_0.wav
Extracted speech token length: tensor([170], device='cuda:0', dtype=torch.int32)
Processing

## 预处理数据加载

In [7]:
from datasets import load_dataset

def load_and_parse_speech_data(json_file):
    """
    使用 datasets 库从 JSON 文件加载数据。
    
    参数:
        json_file (str 或 Path): 包含 speech_tokens 数据的 JSON 文件路径。
        
    返回:
        Dataset: 包含所有处理后的数据条目的 Dataset 对象。
    """
    # 使用 datasets 库加载 JSON 文件
    dataset = load_dataset('json', data_files={'train': str(json_file)}, split='train')

    return dataset

# 使用函数
json_file_path = '/root/project/CosyVoice-main/CosyVoice-main/data/speech_data.json'
loaded_dataset = load_and_parse_speech_data(json_file_path)

2024-12-25 13:16:31,359 INFO PyTorch version 2.3.1 available.
2024-12-25 13:16:31,653 DEBUG Starting new HTTPS connection (1): s3.amazonaws.com:443
2024-12-25 13:16:32,463 DEBUG https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/datasets/json/json.py HTTP/1.1" 200 0
2024-12-25 13:16:32,477 DEBUG Attempting to acquire lock 140035895162912 on /root/.cache/huggingface/datasets/_root_.cache_huggingface_datasets_json_default-63ce0c5360ce48f3_0.0.0_ab573428e7a11a7e23eebd41a2a71665ac3789ce311cbad7049572034a9bda05.lock
2024-12-25 13:16:32,479 DEBUG Lock 140035895162912 acquired on /root/.cache/huggingface/datasets/_root_.cache_huggingface_datasets_json_default-63ce0c5360ce48f3_0.0.0_ab573428e7a11a7e23eebd41a2a71665ac3789ce311cbad7049572034a9bda05.lock
2024-12-25 13:16:32,480 DEBUG open file: /root/.cache/huggingface/datasets/json/default-63ce0c5360ce48f3/0.0.0/ab573428e7a11a7e23eebd41a2a71665ac3789ce311cbad7049572034a9bda05/dataset_info.json
2024-12-25 13:16:32,481 DEBUG Atte

In [6]:
loaded_dataset

Dataset({
    features: ['input', 'output', 'speech_token_len', 'instruction', 'path', 'speech_tokens'],
    num_rows: 100
})

## 模型加载

In [1]:
from hyperpyyaml import load_hyperpyyaml
from cosyvoice.cli.frontend import CosyVoiceFrontEnd
import os
import torch
from cosyvoice.cli.model import CosyVoice2Model
from cosyvoice.utils.file_utils import load_wav
import torchaudio
import sys
device = 'cuda'
sys.path.append('/root/project/GLM-4-Voice-main/GLM-4-Voice-main/third_party/Matcha-TTS')

model_dir = "/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B"
with open('{}/cosyvoice.yaml'.format(model_dir), 'r') as f:
            configs = load_hyperpyyaml(f, overrides={'qwen_pretrain_path': os.path.join(model_dir, 'CosyVoice-BlankEN')})

llm_model = model_dir + "/llm.pt"       
llm = configs['llm']
llm.load_state_dict(torch.load(llm_model, map_location=device), strict=True)

failed to import ttsfrd, use WeTextProcessing instead


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2024-12-23 16:53:39,787 INFO input frame rate=25
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


<All keys matched successfully>

## 原始Qwen和Tokenizer加载

In [6]:
from transformers import AutoTokenizer,AutoModelForCausalLM
path = '/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B/CosyVoice-BlankEN'
tokenizer = AutoTokenizer.from_pretrained(path)
# ori_model = AutoModelForCausalLM.from_pretrained(path)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [13]:
ori_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm()
        (post_attention_layernorm): Qwen2RMSNorm()
      )
    )
    (norm): Qwen2RMSNorm()
  )
  (lm_head): Linear(in_featur

## 将Cosyvoice的权重复制给qwen

In [14]:
llm_model_path = "/root/.cache/modelscope/hub/iic/CosyVoice2-0.5B/llm.pt"       

import torch

def load_matching_params(model, saved_state_dict_path):
    # 加载保存的模型参数
    saved_state_dict = torch.load(saved_state_dict_path)

    # 获取当前模型的状态字典
    model_state_dict = model.state_dict()

    # 初始化一个新的状态字典用于存储匹配的参数
    updated_state_dict = model_state_dict.copy()

    # 定义一个函数来调整原始模型的key以适应新模型
    def adapt_key(key):
        # 如果key以 'llm.model.model' 开头，去掉第一个 'model'
        if key.startswith('llm.model.model'):
            return key.replace('llm.model.model', 'model', 1)
        elif key.startswith('llm.model'):
            return key.replace('llm.model.', '', 1)
        else:
            return key

    # 遍历保存的状态字典
    for name, param in saved_state_dict.items():
        adapted_name = adapt_key(name)
        
        # 如果调整后的键存在于当前模型的状态字典中，并且形状匹配，则更新参数
        if adapted_name in model_state_dict and model_state_dict[adapted_name].shape == param.shape:
            print(f"Loading parameter: {adapted_name}")
            updated_state_dict[adapted_name] = param
        else:
            print(f"Parameter not found or shape mismatch: {adapted_name}. Skipping.")

    # 使用新的状态字典更新模型参数
    model.load_state_dict(updated_state_dict, strict=False)  # 设置strict=False以忽略缺失的或不匹配的参数

# 假设你已经有一个定义好的模型实例 `model`
# 和一个包含预训练参数的文件路径 `path_to_saved_params.pt`

load_matching_params(ori_model, llm_model_path)

Parameter not found or shape mismatch: llm_embedding.weight. Skipping.
Loading parameter: model.embed_tokens.weight
Loading parameter: model.layers.0.self_attn.q_proj.weight
Loading parameter: model.layers.0.self_attn.q_proj.bias
Loading parameter: model.layers.0.self_attn.k_proj.weight
Loading parameter: model.layers.0.self_attn.k_proj.bias
Loading parameter: model.layers.0.self_attn.v_proj.weight
Loading parameter: model.layers.0.self_attn.v_proj.bias
Loading parameter: model.layers.0.self_attn.o_proj.weight
Loading parameter: model.layers.0.mlp.gate_proj.weight
Loading parameter: model.layers.0.mlp.up_proj.weight
Loading parameter: model.layers.0.mlp.down_proj.weight
Loading parameter: model.layers.0.input_layernorm.weight
Loading parameter: model.layers.0.post_attention_layernorm.weight
Loading parameter: model.layers.1.self_attn.q_proj.weight
Loading parameter: model.layers.1.self_attn.q_proj.bias
Loading parameter: model.layers.1.self_attn.k_proj.weight
Loading parameter: model.l

In [2]:
prompt = "请介绍一下北京这座城市"
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(ori_model.device)

In [7]:
generated_ids = ori_model.generate(
    **model_inputs,
    max_new_tokens=100
)
# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

# response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [4]:
response

'北京，简称“北”，是中华人民共和国的首都。位于中国东北部、太行山以南、华北平原西部边缘，东临渤海，西隔门头沟区与河北省相望。北京自古以来就是中国的政治、文化、经济和国际交往中心，也是世界上著名的历史文化名城之一。\n\n北京拥有悠久的历史和丰富的文化遗产，如故宫、长城、颐和园等名胜古迹；还有众多博物馆、图书馆、剧院等文化设施，以及众多大型购物中心和旅游景点，为市民提供了丰富多样的休闲娱乐场所。此外，北京还是全球知名的商业城市之一，吸引了大量国内外游客前来观光旅游。'

## 数据预处理

## 加入special token

In [2]:
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}

In [8]:
tokenizer.add_special_tokens({"additional_special_tokens": ["<|speech_start|>", "<|speech_end|>"]})

2

In [18]:
tokenizer.encode("<|speech_end|>")

[151647]

In [10]:
import numpy as np
def process_func(example,tokenizer):
    """
    将数据集进行预处理
    """
    max_length = 512
    Off_set = 151936
    instruction_str = (
        f"<|im_start|>system\n{example['instruction']}，<|im_end|>\n"
        f"<|im_start|>user\n{example['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    text_labels = (f"{example['output']}<|im_end|>\n")

    instruction_token = tokenizer(instruction_str, add_special_tokens=False)
    text_labels_token = tokenizer(text_labels, add_special_tokens=False)
    # text_token = instruction_token["input_ids"]
    speech_token = [token + Off_set for token in example["speech_tokens"]]

    input_ids =  instruction_token["input_ids"] + text_labels_token['input_ids'] + tokenizer.encode('<|speech_start|>') + speech_token + tokenizer.encode('<|speech_end|>')
    
    # 构建标签序列，只有text_labels部分的token可以作为标签
    labels = (
        [0] * len(instruction_token["input_ids"]) +  # 指令部分不作为标签
        [1] * len(text_labels_token['input_ids']) +   # 标签部分作为标签
        [1] * (len(speech_token) + 2)  # 特殊标记和语音token不作为标签
    )

    # if len(input_ids+speech_labels) > max_length:
    #     input_ids = input_ids[:max_length]
    #     labels = labels[:max_length]

    return {
        "input_ids": input_ids,
        "labels": labels
    }

In [11]:
from functools import partial
func = partial(process_func, tokenizer=tokenizer)


In [12]:
process_ds = loaded_dataset.map(func)

In [14]:
from torch.utils.data import DataLoader
import torch


def collate_fn(batch):
    pad_token_id = 151643 
    IGNORE_ID = -1
    max_length = min(max(len(item['input_ids']) for item in batch), 512)

    # 动态填充和截断 input_ids 和 labels
    input_ids = []
    true_labels = []

    for item in batch:
        # 截取或填充 input_ids 到最大长度
        item_input_ids = item['input_ids'][:max_length]
        item_input_ids += [pad_token_id] * (max_length - len(item_input_ids))
        
        # 根据 labels 确定哪些 token 是有效的（即 label 为 1）
        item_labels = item['labels'][:max_length]
        item_true_labels = [item_input_ids[i+1] if label == 1 else IGNORE_ID for i, label in enumerate(item_labels[:-1])]
        item_true_labels.append(pad_token_id)  # 最后一个位置总是填充token
        
        # 填充到最大长度
        item_true_labels += [pad_token_id] * (max_length - len(item_true_labels))

        input_ids.append(item_input_ids)
        true_labels.append(item_true_labels)

    # 将列表转换为张量
    input_ids = torch.tensor(input_ids)
    true_labels = torch.tensor(true_labels)

    return {
        'input_ids': input_ids,
        'labels': true_labels
    }



In [15]:
# 创建 DataLoader 并传入自定义的 collate_fn
dataloader = DataLoader(process_ds, batch_size=1, num_workers=1, shuffle=False, collate_fn=collate_fn)
i, x = next(enumerate(dataloader))

In [8]:
import torch
device = 'cuda'
for i,batch in enumerate(dataloader):
    input_ids = batch['input_ids']
    labels = batch['labels']
    
    # attention_mask = batch['attention_mask']
    output = llm(input_ids,labels)
    print(f"input_ids: {input_ids.shape}\n, labels: {labels.shape}\n")
    

: 

## 模型训练

## 模型保存

## 模型测试

In [7]:
prompt = "你是一个智能语音助手，请根据用户的问题给出回答"
text = '你是谁？'

instruction_str = (
        f"<|im_start|>system\n{prompt}，<|im_end|>\n"
        f"<|im_start|>user\n{text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

instruction = tokenizer(instruction_str, add_special_tokens=False)["input_ids"]
input = torch.tensor(instruction).view(1, -1)
input

tensor([[151644,   8948,    198,  56568,  20412,  14777,  18947,  99473,  26232,
          72881,  78685,  99262,  44934,  37945,  99408,  16038,  11622,  17523,
           9370,  56007,  33872,  89012,  20221,  18397,  99590,   3837, 151645,
            198, 151644,    872,    198,  56568,  20412, 100165,  11319, 151645,
            198, 151644,  77091,    198]])

In [11]:
torch.tensor(instruction).view(1, -1)

tensor([[151644,   8948,    198,  56568,  20412,  14777,  18947,  99473,  26232,
          72881,  78685,  99262,  44934,  37945,  99408,  16038,  11622,  17523,
           9370,  56007,  33872,  89012,  20221,  18397,  99590,   3837, 151645,
            198, 151644,    872,    198,  56568,  20412, 100165, 151645,    198,
         151644,  77091,    198]])

## Only Speech input

In [12]:
tts_speech_token_dict = []
for i in llm.inference(text=torch.zeros(1,0, dtype=speech.dtype).to(device),
                        text_len=0,
                        prompt_text=torch.zeros(1,0, dtype=speech.dtype).to(device),
                        prompt_text_len=0,
                        prompt_speech_token=torch.zeros(1,0, dtype=speech.dtype).to(device),#torch.zeros(1, 0, dtype=speech.dtype).to(device),
                        prompt_speech_token_len=torch.tensor([speech.shape[1]], dtype=torch.int32).to(device),
                        embedding=0):

        tts_speech_token_dict.append(i)


## Only Text input

In [8]:
tts_speech_token_dict = []
device = 'cuda'
for i in llm.inference(text=input.to(device),
                        text_len=torch.tensor([input.shape[1]], dtype=torch.int32).to(device),
                        prompt_text=torch.zeros(1,0, dtype=input.dtype).to(device),
                        prompt_text_len=0,
                        prompt_speech_token=torch.zeros(1,0, dtype=input.dtype).to(device),#torch.zeros(1, 0, dtype=speech.dtype).to(device),
                        prompt_speech_token_len=0,
                        embedding=torch.zeros(1,0, dtype=input.dtype).to(device)):
        tts_speech_token_dict.append(i)


In [9]:
tts_speech_token_dict

[3704,
 4137,
 6324,
 4137,
 1950,
 3161,
 5194,
 5832,
 5832,
 5832,
 3645,
 3672,
 3888,
 3888,
 3645,
 3645,
 1704,
 4137,
 1959,
 3747,
 3744,
 4456,
 6157,
 6400,
 6076,
 5186,
 2228,
 3455,
 3320,
 1208,
 716,
 710,
 1454,
 2181,
 2112,
 324,
 2592,
 3240,
 5015,
 5006,
 4763,
 4603,
 1113,
 4918,
 2262,
 5586,
 4778,
 2264,
 1044,
 2261,
 704,
 728,
 2159,
 2159,
 2159,
 1457,
 1448,
 710,
 602,
 575,
 575,
 1223,
 4218,
 6324,
 6324,
 6324,
 6324,
 4137,
 1950,
 324,
 2349,
 2520,
 4772,
 632,
 2567,
 4610,
 4777,
 5749,
 5668,
 5668,
 3482,
 1295,
 1295,
 1295,
 2104,
 1852,
 4218,
 6324,
 6324,
 6324,
 6324,
 6405,
 4353,
 469,
 4594,
 4517,
 2339,
 2591,
 2726,
 350,
 1806,
 4156,
 1241,
 1268,
 1106,
 1808,
 6178,
 5529,
 2139,
 297,
 4456,
 4133,
 1946,
 974,
 1035,
 6351,
 5644,
 2826,
 5338,
 5342,
 2348,
 807,
 2085,
 4674,
 3319,
 4777,
 2261,
 3961,
 65,
 2567,
 1206,
 2112,
 4299,
 1917,
 5587,
 4778,
 4442,
 4433,
 4423,
 4477,
 4560,
 4416,
 4509,
 4431,
 4432,
 22

In [19]:
tokenizer.decode(input.squeeze(0).tolist())

'你是一个智能语音助手，请根据用户的问题给出回答,你会做规划吗？'

In [11]:
tokenizer.decode(text_token_dict)

"文感，高科学信教我不奇文然足。感高科<|speech_start|><|speech_start|>。<|speech_start|>资草然代高说绘مناقش、的舞高照足溦考足造草打足草要没，不.Closed chapters足字能短дум具草模具体字的聊生然足没、我手草足高学造要信结没、不然短我足。生程 Nan字草 going花感的高足。造信学草不没，分手生高高短Great足。字武要生，结无然为草感足造报球没高 short的然考草，我说的造科高信。没动 valueForKey足不不，音说来 c的教然我没足草球音的高，学然我抱美容高手考_po。 going的我音人 al玩高然流生手。足没我球生足然足视武生信的。学高 '';\n友聊来考闻不效短我结科的学流我武能足试手然来高考、。感足造闻没，手的学我文无聊足。球手、字讲武，我信我高学ting生。来球短的我高草场活信生面下。的学闻足学我聊，高手为足说感㏜我来考音学 It考高手生的学我说足球。流有字球结手 going音不能生学感的学舞高足草球我来人高，订讲 Mont。能我旗实小高足结 going视生流短学的年不我校期。场然 To going的verte足高 not球生ants我字 going。然文，下草科 cl的我球聊学高足。字Deleted短足球学的技，"

In [45]:
import uuid
this_uuid = str(uuid.uuid1())
model.hift_cache_dict[this_uuid] = None
resample_rate = 24000

input_speech_16k = load_wav('/root/project/CosyVoice-main/CosyVoice-main/template/rap.wav', 16000)
input_text = '我的话也说的好听人也平和谦逊,不需要华而不实的文凭,在这里无需澄清,对号入座的把扶手都抓紧，我深得民心,不是我乐意我也不想煞了一个夏天的风景,所憧憬所恭敬的人却配不上这虚伪拙劣的崇敬'
input_speech_resample = torchaudio.transforms.Resample(orig_freq=16000, new_freq=resample_rate)(input_speech_16k)
input_feat, input_feat_len = frontend._extract_speech_feat(input_speech_resample)
input_token, input_token_len = frontend._extract_speech_token(input_speech_16k)


prompt_speech_16k = load_wav('/root/project/CosyVoice-main/CosyVoice-main/template/fine_grained_control_0.wav', 16000)
prompt_text = '在他讲述那个荒诞故事的过程中，他突然停下来，因为他自己也被逗笑了。'
prompt_speech_resample = torchaudio.transforms.Resample(orig_freq=16000, new_freq=resample_rate)(prompt_speech_16k)
speech_feat, speech_feat_len = frontend._extract_speech_feat(prompt_speech_resample)
speech_token, speech_token_len = frontend._extract_speech_token(prompt_speech_16k)

# if resample_rate == 24000:
#     # cosyvoice2, force speech_feat % speech_token = 2
#     token_len = min(int(speech_feat.shape[1] / 2), speech_token.shape[1])
#     speech_feat, speech_feat_len[:] = speech_feat[:, :2 * token_len], 2 * token_len
#     speech_token, speech_token_len[:] = speech_token[:, :token_len], token_len
embedding = frontend._extract_spk_embedding(input_speech_16k)


this_tts_speech = model.token2wav(token=speech_token,#torch.tensor(tts_speech_token_dict).view(1,-1),
                                 prompt_token=input_token,#speech_token,
                                 prompt_feat=input_feat,#speech_feat,
                                 embedding=embedding,
                                 uuid=this_uuid,
                                 token_offset=0,
                                 finalize=True,
                                 speed=1)
torchaudio.save('/root/project/CosyVoice-main/CosyVoice-main/template/trained-2.wav', this_tts_speech.cpu(), 24000)

In [32]:
embedding.shape

torch.Size([1, 192])

In [48]:
speech_token.shape

torch.Size([1, 189])

In [49]:
input_feat.shape

torch.Size([1, 690, 80])

In [43]:
torch.save(input_feat,"/root/project/CosyVoice-main/speech_feat.pt")

In [44]:
torch.save(embedding,"/root/project/CosyVoice-main/embedding.pt")

In [46]:
input_token.shape

torch.Size([1, 345])

In [47]:
input_feat.shape

torch.Size([1, 690, 80])